<a href="https://colab.research.google.com/github/parshav42/50_ML_models/blob/main/sugarcanemodel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!unzip -b /content/drive/MyDrive/sugarcane/Sugarcane_leafs1.zip -d  /content

In [ ]:
import torch

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available else 'cpu')


In [ ]:
import os
import random
import shutil

image_folder = "/content/Sugarcane_leafs/Yellow"
# label_folder = "/content/sugarcaneimge/labels"
output_folder = "/content/dataset"

random.seed(42)

images = [
    f for f in os.listdir(image_folder)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

random.shuffle(images)

split = int(len(images) * 0.5)
train_images = images[:split]
val_images = images[split:]

for folder in [
    "images/train",
    "images/val",
    # "labels/train",
    # "labels/val"
]:
    os.makedirs(os.path.join(output_folder, folder), exist_ok=True)

def copy_files(image_list, split_name):
    for image in image_list:
        name = os.path.splitext(image)[0]

        shutil.copy(
            os.path.join(image_folder, image),
            os.path.join(output_folder, "images", split_name, image)
        )

        # label = name + ".txt"
        # label_path = os.path.join(label_folder, label)

        # if os.path.exists(label_path):
        #     shutil.copy(
        #         label_path,
        #         os.path.join(output_folder, "labels", split_name, label)
        #     )

copy_files(train_images, "train")
copy_files(val_images, "val")

print("Train:", len(train_images))
print("Validation:", len(val_images))

In [ ]:
!rm -rf '/content/Sugarcane_leafs'

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('/content/dataset', 'zip', '/content/dataset')
# files.download('/content/dataset.zip')

In [ ]:
image_path = '/content/dataset'

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

img = Image.open('/content/dataset/test/healthy/cropped_healthy (110).jpeg')
plt.imshow(img)
plt.axis('off') # Hides the pixel coordinate axes
plt.show()


In [ ]:
from torchvision import datasets,transforms
from torch.utils.data import DataLoader

In [ ]:
transf = transforms.Compose([
    transforms.Resize([224,224]),
    transforms.ToTensor(),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

In [ ]:
from pathlib import Path

# Setup path to data folder
data_path = Path("/content/dataset")
train_img = data_path / "train"
test_img = data_path / "test"

In [ ]:
train_img , test_img

In [ ]:
import shutil
from pathlib import Path

checkpoint_path = data_path / "train" / ".ipynb_checkpoints"
checkpoint_path1 = data_path / "test" / ".ipynb_checkpoints"

if checkpoint_path.exists():
    shutil.rmtree(checkpoint_path)
if checkpoint_path1.exists():
    shutil.rmtree(checkpoint_path1)

In [ ]:
train_data = datasets.ImageFolder(
    root = data_path /  "train",
    transform = transf,
    target_transform = None


)
test_data = datasets.ImageFolder(
    root=data_path / "test",
    transform=val_transform
)


In [ ]:
print(len(train_data))

In [ ]:
import matplotlib.pyplot as plt

image, label = train_data[8790]

# Image is a tensor: [C, H, W]
plt.imshow(image.permute(1, 2, 0))
plt.title(f"Label: {train_data.classes[label]}")
plt.axis("off")
plt.show()

In [ ]:
print(train_data.classes)

In [ ]:
class_dic = train_data.class_to_idx
class_dic

In [ ]:
len(test_data),len(train_data)

In [ ]:
train_datalo = DataLoader(
    dataset = train_data,
    batch_size =32,
    shuffle = True,
    num_workers=0,
    pin_memory=True,



)

test_datalo = DataLoader(
    dataset = test_data,
    batch_size = 32,
    shuffle = False
)

In [ ]:
img, label = next(iter(train_datalo))

# Batch size will now be 1, try changing the batch_size parameter above and see what happens
print(f"Image shape: {img.shape} -> [batch_size, color_channels, height, width]")
print(f"Label shape: {label.shape}")

In [ ]:
from torch import nn

class TinyVGG(nn.Module):
    """
    Model architecture copying TinyVGG from:
    https://poloclub.github.io/cnn-explainer/
    """
    def __init__(self, input_shape: int, hidden_units: int, output_shape: int) -> None:
        super().__init__()
        self.conv_block_1 = nn.Sequential(
            nn.Conv2d(in_channels=input_shape,
                      out_channels=hidden_units,
                      kernel_size=3, # how big is the square that's going over the image?
                      stride=1, # default
                      padding=1), # options = "valid" (no padding) or "same" (output has same shape as input) or int for specific number
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units,
                      out_channels=hidden_units,
                      kernel_size=3,
                      stride=1,
                      padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,
                         stride=2) # default stride value is same as kernel_size
        )
        self.conv_block_2 = nn.Sequential(
            nn.Conv2d(hidden_units, hidden_units, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(hidden_units, hidden_units, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            # Where did this in_features shape come from?
            # It's because each layer of our network compresses and changes the shape of our input data.
            # Corrected in_features calculation based on 128x128 input and two 2x2 max pooling layers
            nn.Linear(in_features=hidden_units*56*56,
                      out_features=output_shape)
        )

    def forward(self, x: torch.Tensor):
        x = self.conv_block_1(x)
        # print(x.shape) # For debugging: torch.Size([batch_size, 10, 64, 64])
        x = self.conv_block_2(x)
        # print(x.shape) # For debugging: torch.Size([batch_size, 10, 32, 32])
        x = self.classifier(x)
        # print(x.shape)
        return x
        # return self.classifier(self.conv_block_2(self.conv_block_1(x))) # <- leverage the benefits of operator fusion

torch.manual_seed(42)
model = TinyVGG(input_shape=3, # number of color channels (3 for RGB)
                  hidden_units=10,
                  output_shape=len(train_data.classes))
model

In [ ]:
# model = model.to(device)
loss_fn = nn.CrossEntropyLoss()

optim = torch.optim.Adam(model.parameters(),lr=0.001)

In [ ]:
epochs = 100
for epoch in range(epochs):

  total_epoch_loss = 0

  for batch_features, batch_labels in train_datalo:


    # batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

    # forward pass
    outputs = model(batch_features)

    # calculate loss
    loss = loss_fn(outputs, batch_labels)

    # back pass
    optim.zero_grad()
    loss.backward()

    # update grads
    optim.step()

    total_epoch_loss = total_epoch_loss + loss.item()

  avg_loss = total_epoch_loss/len(train_datalo)
  print(f'Epoch: {epoch + 1} , Loss: {avg_loss}')



In [ ]:
model.eval()

In [ ]:
total = 0
correct = 0

with torch.no_grad():

  for batch_features, batch_labels in test_datalo:

    # batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

    outputs = model(batch_features)

    _, predicted = torch.max(outputs, 1)

    total = total + batch_labels.shape[0]

    correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)